## ** ASSIGNMENT 2-  BIG DATA SYSTEMS (S1-25_DSECLZG522)**
##** Group Number: 34
### Group Member Names
1. **Samarth Mohanrao** Kulkarni | ID: 2024DA04090 | Contribution: 100%
2. **RANGANATHA S** | ID: 2024da04087 | Contribution: 100%
3. **BANDARU HAREESHA** | ID: 2024da04089 | Contribution: 100%

---

## Dataset Used for Implementation
**Title:** Amazon Product Review Analysis using Apache Spark

**Dataset:** Amazon Product Reviews dataset publicly
available real-world corpus

## Dataset Description
Amazon Product reviews in publicly availabe data a realworld dataset.

## Objective :
The primary objective of this assignment is to gain practical, hands-on experience in leveraging Apache Spark for large-scale data processing and analysis. By utilizing the Amazon Product Reviews dataset

## 1. Environment Setup - Install Java and Download Spark

In [ ]:
# Local Setup - Java 17 is already installed
import os
import sys

java_home = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'
os.environ['JAVA_HOME'] = java_home
print(f'✓ JAVA_HOME set to Java 17')

# Install findspark
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'findspark'], check=False)
print('✓ Environment ready for Spark')

zsh:1: command not found: apt-get
zsh:1: command not found: apt-get
zsh:1: command not found: wget
tar: Error opening archive: Failed to open 'spark-3.5.0-bin-hadoop3.tgz'
zsh:1: command not found: pip


In [ ]:
# Initialize PySpark
import os
import findspark
from pyspark.sql import SparkSession

java_home = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'
os.environ['JAVA_HOME'] = java_home

findspark.init()

spark = SparkSession.builder \
    .appName('BDS-Assignment') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

print(f'✓ Spark {spark.version} initialized')

Trying to download Spark from: https://dlcdn.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz


FileNotFoundError: [Errno 2] No such file or directory: 'wget'

In [ ]:
#Step 3 -  Initialize PySpark and run a quick test
import os, findspark
# Ensure env vars are set for this session (in case the runtime refreshed)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# If your SPARK_HOME path prints None, replace it with the folder name you saw (e.g., /content/spark-3.5.0-bin-hadoop3)
if "SPARK_HOME" not in os.environ or not os.path.exists(os.environ["SPARK_HOME"]):
    # Auto-detect common location in Colab
    for d in os.listdir("/content"):
        if d.startswith("spark-3.5.0-bin-hadoop3"):
            os.environ["SPARK_HOME"] = f"/content/{d}"
            break
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F



JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: /content/spark-3.5.0-bin-hadoop3


##St1.Data loading:
Load the dataset into a Spark with schema inference as
appropriate. Print the schema and the number of records loaded.

In [ ]:
# Data file path (local)
data_path = '/Users/hareeshabandaru/Documents/BITS_WILP_MTECH_DS/Assignments/Semester3/BDS/AmazonProductReviews.csv'
print(f'✓ Data path set: {data_path}')

Saving AmazonProductReviews.csv to AmazonProductReviews.csv


In [ ]:
print('✓ Spark session ready')

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: /content/spark-3.5.0-bin-hadoop3
PySpark version: 3.5.0


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DateType

df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .option('multiLine', True)
    .option('escape', '"')
    .csv(data_path)
)

print(f'✓ Loaded {df.count()} records')
df.printSchema()
df.show(5, truncate=False)

Rows: 1597
root
 |-- id: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- colors: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- dimension: string (nullable = true)
 |-- ean: double (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- name: string (nullable = true)
 |-- prices: string (nullable = true)
 |-- reviews.date: timestamp (nullable = true)
 |-- reviews.doRecommend: boolean (nullable = true)
 |-- reviews.numHelpful: integer (nullable = true)
 |-- reviews.rating: integer (nullable = true)
 |-- reviews.sourceURLs: string (nullable = true)
 |-- reviews.text: string (nullable = true)
 |-- reviews.title: string (nullable = true)
 |-- reviews.userCity: string (nullable = true)
 |-- reviews.userProvince: string (nullabl

##Step 2.Data cleansing:
Make necessary changes in Schema, as required for
the given analytical queries. Create a new column with name
“primary_category” and populate it with the first category present in
the ‘categories’ column. Drop rows where rating is missing or outside
the valid range (1–5). List the modified schema and find the number
of records after dropping the rows.

In [ ]:
#Data cleansing
from pyspark.sql import functions as F

# Helper to safely reference columns that contain dots
def c(name: str):
    return F.col(f"`{name}`")

# 1) Create primary_category (take the first category before a comma)
df_clean = df.withColumn(
    "primary_category",
    F.split(c("categories"), ",").getItem(0)
)

# 2) Ensure reviews.rating is numeric 1–5
df_clean = df_clean.withColumn(
    "rating_num",
    c("reviews.rating").cast("double")
)

before_cnt = df_clean.count()

df_clean = df_clean.filter(
    (F.col("rating_num").isNotNull()) &
    (F.col("rating_num") >= 1.0) &
    (F.col("rating_num") <= 5.0)
)

after_cnt = df_clean.count()

# 3) Parse reviews.date to proper date (try multiple common formats)
possible_formats = ["yyyy-MM-dd", "MM/dd/yyyy", "dd-MMM-yy", "yyyy/MM/dd"]
parsed = None
for fmt in possible_formats:
    candidate = F.to_date(c("reviews.date"), fmt)
    parsed = candidate if parsed is None else F.coalesce(parsed, candidate)

df_clean = df_clean.withColumn("reviews_date_parsed", parsed)

# 4) Select & rename for a cleaner schema
df_clean = df_clean.select(
    c("categories"),
    F.col("primary_category"),
    c("name"),
    c("reviews.title").alias("reviews.title"),
    c("reviews.text").alias("reviews.text"),
    F.col("rating_num").alias("reviews.rating"),
    c("reviews.username").alias("reviews.username"),
    F.col("reviews_date_parsed").alias("reviews.date")
)

print(f"Records before rating filter: {before_cnt}")
print(f"Records after rating filter : {after_cnt}")
df_clean.printSchema()
df_clean.show(5, truncate=False)

Records before rating filter: 1597
Records after rating filter : 1177
root
 |-- categories: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- reviews.title: string (nullable = true)
 |-- reviews.text: string (nullable = true)
 |-- reviews.rating: double (nullable = true)
 |-- reviews.username: string (nullable = true)
 |-- reviews.date: date (nullable = true)

+--------------------------+----------------+-----------------+------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

##Step-3 Top Products by Average Rating with Minimum Reviews:
  Find product
names with at least 20 reviews and rank them by average rating.

In [ ]:
#Step 6: Top Products by Average Rating (min. 20 reviews)
from pyspark.sql import functions as F

# Create a query-friendly view with simple column names
df_q = (
    df_clean
    .withColumnRenamed("reviews.rating", "rating")
    .withColumnRenamed("reviews.date", "review_date")
    .withColumnRenamed("reviews.title", "review_title")
    .withColumnRenamed("reviews.text", "review_text")
    .withColumnRenamed("reviews.username", "username")
)

df_q.printSchema()  # quick sanity check; look for 'rating' (double)

min_reviews = 20

top_products = (
    df_q
    .filter(F.col("name").isNotNull())
    .groupBy("name")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
    .filter(F.col("review_count") >= min_reviews)
    .orderBy(F.desc("avg_rating"), F.desc("review_count"), F.asc("name"))
)

top_products.show(20, truncate=False)

root
 |-- categories: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- username: string (nullable = true)
 |-- review_date: date (nullable = true)

+-----------------------------------------------------+------------+----------+
|name                                                 |review_count|avg_rating|
+-----------------------------------------------------+------------+----------+
|Fire HD 6 Tablet                                     |38          |5.0       |
|Kindle Paperwhite                                    |22          |4.59      |
|Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker|542         |4.53      |
|Kindle Fire HDX 7"                                   |23          |4.39      |
|Amazon Fire TV                                       |44          |4.2       |
|Amazon Premium Headph

## Step 4 - Most Active Reviewers (by Review Count):
 List top 10 most active
reviewers.

In [ ]:
#Most Active Reviewers (Top 10 by Review Count)
from pyspark.sql import functions as F

top_reviewers = (
    df_q
    .filter(F.col("username").isNotNull())
    .groupBy("username")
    .agg(F.count("*").alias("review_count"))
    .orderBy(F.desc("review_count"))
    .limit(10)
)

top_reviewers.show(truncate=False)

+---------------+------------+
|username       |review_count|
+---------------+------------+
|A. Younan      |38          |
|Andrew         |23          |
|William Hardin |23          |
|Amazon Customer|17          |
|Victor L.      |15          |
|Earthling1984  |13          |
|NF             |13          |
|Amazon Reviewer|12          |
|Mike W.        |12          |
|D. Miyao       |10          |
+---------------+------------+



##Step 5
Show how ratings evolve over time per category:  Show the Monthly
Trend of Average Ratings per primary_category of product. Show 40
rows.

In [ ]:
#Step 8: Monthly Trend of Average Ratings per Primary Category (40 rows)
from pyspark.sql import functions as F

monthly_trend = (
    df_q
    .withColumn("year_month", F.date_format(F.col("review_date"), "yyyy-MM"))
    .groupBy("primary_category", "year_month")
    .agg(F.round(F.avg("rating"), 2).alias("avg_rating"))
    .orderBy("primary_category", "year_month")
    .limit(40)
)

monthly_trend.show(40, truncate=False)


+----------------------------+----------+----------+
|primary_category            |year_month|avg_rating|
+----------------------------+----------+----------+
|Amazon Devices              |NULL      |4.15      |
|Amazon Devices              |2012-09   |4.5       |
|Amazon Devices              |2012-10   |4.0       |
|Amazon Devices              |2013-10   |4.31      |
|Amazon Devices              |2013-11   |4.17      |
|Amazon Devices              |2013-12   |5.0       |
|Amazon Devices              |2014-07   |5.0       |
|Amazon Devices              |2014-09   |4.0       |
|Amazon Devices              |2014-10   |3.67      |
|Amazon Devices              |2014-11   |5.0       |
|Amazon Devices              |2015-03   |1.0       |
|Amazon Devices              |2015-04   |5.0       |
|Amazon Devices              |2015-06   |5.0       |
|Amazon Devices              |2015-07   |4.8       |
|Amazon Devices              |2015-08   |5.0       |
|Amazon Devices              |2015-09   |5.0  

## Step 6 Find products loved by some and hated by few:
 List top 10 Product
names by the Ratio of 5-Star to 1-Star Reviews.

In [ ]:
#Products loved by some and hated by few (Top 10 by 5★/1★ ratio)

from pyspark.sql import functions as F

# Aggregate 5★ and 1★ counts per product
stars_agg = (
    df_q
    .filter(F.col("name").isNotNull())
    .groupBy("name")
    .agg(
        F.sum(F.when(F.col("rating") == 5, 1).otherwise(0)).alias("five_star"),
        F.sum(F.when(F.col("rating") == 1, 1).otherwise(0)).alias("one_star"),
        F.count("*").alias("review_count")
    )
)

# Compute ratio; require a minimum review volume (optional but useful). You can change 10 → any threshold.
min_reviews_for_ratio = 10
ratio_df = (
    stars_agg
    .withColumn("ratio", F.col("five_star") / (F.col("one_star") + F.lit(1)))
    .filter(F.col("review_count") >= min_reviews_for_ratio)
    .orderBy(F.desc("ratio"), F.desc("five_star"), F.asc("name"))
    .limit(10)
)

ratio_df.show(truncate=False)

+---------------------------------------------------------------------------+---------+--------+------------+-----+
|name                                                                       |five_star|one_star|review_count|ratio|
+---------------------------------------------------------------------------+---------+--------+------------+-----+
|Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker                      |358      |7       |542         |44.75|
|Amazon Premium Headphones                                                  |39       |0       |77          |39.0 |
|Fire HD 6 Tablet                                                           |38       |0       |38          |38.0 |
|Kindle Paperwhite                                                          |15       |0       |22          |15.0 |
|All-New Amazon Kid-Proof Case for Amazon Fire HD 8 Tablet (7th Generation  |12       |0       |12          |12.0 |
|All-New Amazon Kid-Proof Case for Amazon Fire 7 Tablet (7th Generation 

## Step 7 - Longest Review Texts per Category:
List the longest review in each
primary category along with review title and length of review text. List
it in the order of review length. For this question, result can be
displayed with “truncate=True” as the reviews can be very long.

In [ ]:
#Longest Review Text per Primary CategoryStep 10
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Add a length column for the review text
df_len = df_q.withColumn("review_length", F.length(F.col("review_text")))

# Window per primary_category, ordered by review_length descending
w = Window.partitionBy("primary_category").orderBy(F.desc("review_length"))

# Pick the longest (row_number == 1) review per category
longest_reviews = (
    df_len
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "primary_category",
        "name",
        "review_title",
        "review_text",
        "review_length"
    )
    .orderBy(F.desc("review_length"))
)

# Show results; truncate allowed per assignment
longest_reviews.show(50, truncate=True)


+--------------------+--------------------+--------------------+--------------------+-------------+
|    primary_category|                name|        review_title|         review_text|review_length|
+--------------------+--------------------+--------------------+--------------------+-------------+
|          Categories|      Amazon Fire TV|This box is a GAM...|I am not a casual...|        19739|
|      Amazon Devices|  Kindle Fire HDX 7"|Excellent 3rd-gen...|This is the middl...|        18667|
|Amazon Devices & ...|Alexa Voice Remot...|Great range, very...|As other reviewer...|         1925|
|         Electronics|All-New Fire HD 8...|Fantastic tablet ...|Let me start by s...|         1778|
|        Kindle Store|     Kindle Keyboard|Worth the money. ...|The Kindle is my ...|         1672|
|Cell Phones & Acc...|Moshi Anti-Glare ...|Moshi's screen pr...|I like Moshi's an...|         1379|
+--------------------+--------------------+--------------------+--------------------+-------------+


##Step 8 Growth in review volume:
Show the Year-over-Year Growth in Review
Counts.  

In [ ]:
#Year‑over‑Year (YoY) Growth in Review Counts
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Extract year from the parsed review_date
yearly = (
    df_q
    .filter(F.col("review_date").isNotNull())
    .withColumn("year", F.year(F.col("review_date")))
    .groupBy("year")
    .agg(F.count("*").alias("review_count"))
    .orderBy("year")
)

# Window ordered by year to compute lag
w_year = Window.orderBy("year")

yoy = (
    yearly
    .withColumn("prev_count", F.lag("review_count").over(w_year))
    .withColumn("yoy_count_change", F.col("review_count") - F.col("prev_count"))
    .withColumn(
        "yoy_growth_pct",
        F.when(F.col("prev_count").isNull(), None)
         .when(F.col("prev_count") == 0, None)
         .otherwise(F.round((F.col("yoy_count_change") / F.col("prev_count")) * 100, 2))
    )
)

yoy.show(50, truncate=False)

+----+------------+----------+----------------+--------------+
|year|review_count|prev_count|yoy_count_change|yoy_growth_pct|
+----+------------+----------+----------------+--------------+
|2012|5           |NULL      |NULL            |NULL          |
|2013|24          |5         |19              |380.0         |
|2014|101         |24        |77              |320.83        |
|2015|18          |101       |-83             |-82.18        |
|2016|328         |18        |310             |1722.22       |
|2017|484         |328       |156             |47.56         |
+----+------------+----------+----------------+--------------+



## Step 9 - Average Rating by Review Length buckets:
Analyzes whether longer
reviews tend to be more positive or negative. Use 3 buckets as given
below:

In [ ]:
#Average Rating by Review Length Buckets (Short / Medium / Long)
from pyspark.sql import functions as F

# Create length, bucketize, and aggregate
length_buckets = (
    df_q
    .withColumn("review_len", F.length(F.coalesce(F.col("review_text"), F.lit(""))))
    .withColumn(
        "len_bucket",
        F.when(F.col("review_len") < 50, "Short")
         .when((F.col("review_len") >= 50) & (F.col("review_len") <= 200), "Medium")
         .otherwise("Long")
    )
    .groupBy("len_bucket")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
    # Order buckets logically: Short, Medium, Long
    .withColumn(
        "bucket_order",
        F.when(F.col("len_bucket") == "Short", 1)
         .when(F.col("len_bucket") == "Medium", 2)
         .otherwise(3)
    )
    .orderBy("bucket_order")
    .select("len_bucket", "review_count", "avg_rating")
)

length_buckets.show(truncate=False)

+----------+------------+----------+
|len_bucket|review_count|avg_rating|
+----------+------------+----------+
|Short     |35          |4.57      |
|Medium    |447         |4.48      |
|Long      |695         |4.27      |
+----------+------------+----------+



##Step 10 - Products with Declining Ratings Over Time:
Identify the top 10 product
names whose ratings have dropped maximum. Use monthly average
rating (first and last averages) of a product to identify its rating drop.

In [ ]:
#Products with maximum decline (first vs last monthly average)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1) Monthly average rating per product
monthly_avg = (
    df_q
    .filter(F.col("review_date").isNotNull() & F.col("name").isNotNull())
    .withColumn("year_month", F.date_trunc("month", F.to_timestamp("review_date")))
    .groupBy("name", "year_month")
    .agg(F.round(F.avg("rating"), 3).alias("avg_rating"))
)

# 2) For each product, get FIRST and LAST month + averages
w_full = Window.partitionBy("name").orderBy("year_month") \
               .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

decline_calc = (
    monthly_avg
    .withColumn("first_month", F.first("year_month").over(w_full))
    .withColumn("last_month",  F.last("year_month").over(w_full))
    .withColumn("first_avg",   F.first("avg_rating", ignorenulls=True).over(w_full))
    .withColumn("last_avg",    F.last("avg_rating",  ignorenulls=True).over(w_full))
    .select("name", "first_month", "last_month", "first_avg", "last_avg")
    .dropDuplicates(["name"])
    .withColumn("rating_drop", F.round(F.col("first_avg") - F.col("last_avg"), 3))
    .filter(F.col("rating_drop") > 0)  # only products that actually declined
    .orderBy(F.desc("rating_drop"), F.asc("name"))
    .limit(10)
)

decline_calc.show(truncate=False)

+-----------------------------------------------------------------------------------------+-------------------+-------------------+---------+--------+-----------+
|name                                                                                     |first_month        |last_month         |first_avg|last_avg|rating_drop|
+-----------------------------------------------------------------------------------------+-------------------+-------------------+---------+--------+-----------+
|Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders|2017-03-01 00:00:00|2017-07-01 00:00:00|4.75     |1.0     |3.75       |
|Kindle Fire HDX 8.9"                                                                     |2013-11-01 00:00:00|2015-03-01 00:00:00|4.0      |1.0     |3.0        |
|Alexa Voice Remote for Amazon Echo and Echo Dot                                          |2016-05-01 00:00:00|2016-12-01 00:00:00|3.0      |1.0     |2.0        |
|Kindle for Kids Bundl

## Step 11 - Declining Product Review
Products with Declining Ratings Over Time - Analysis: From on the
results of the above question (top products whose ratings dropped
maximum), select one product and do a data-driven analysis of the
decline and provide your finding and recommendations (max 150
words).

In [ ]:
# Selecting Amazon Tap as product to analyze
product_name = "Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

assert product_name and isinstance(product_name, str), "Please set product_name to a non-empty string."

prod_df = df_q.filter(F.col("name") == product_name)

# Monthly metrics per product
monthly = (
    prod_df
    .filter(F.col("review_date").isNotNull())
    .withColumn("ym", F.date_trunc("month", F.to_timestamp("review_date")))
    .groupBy("ym")
    .agg(
        F.count("*").alias("reviews"),
        F.round(F.avg("rating"), 3).alias("avg_rating"),
        F.sum(F.when(F.col("rating") == 1, 1).otherwise(0)).alias("one_star"),
        F.sum(F.when(F.col("rating") == 5, 1).otherwise(0)).alias("five_star")
    )
    .orderBy("ym")
)

if monthly.count() == 0:
    print(f"No dated reviews found for: {product_name}")
else:
    rows = monthly.collect()
    n = len(rows)

    first_month = rows[0]["ym"]
    last_month  = rows[-1]["ym"]
    first_avg   = rows[0]["avg_rating"]
    last_avg    = rows[-1]["avg_rating"]
    rating_drop_val = (float(first_avg) - float(last_avg)) if (first_avg is not None and last_avg is not None) else None

    # Early vs Late windows (up to 3 months each side)
    k = max(1, min(3, n // 2 if n >= 2 else 1))
    early = rows[:k]
    late  = rows[-k:]

    def sums(block):
        r = sum(int(x["reviews"]) for x in block)
        o = sum(int(x["one_star"]) for x in block)
        f = sum(int(x["five_star"]) for x in block)
        return r, o, f

    e_rev, e_1s, e_5s = sums(early)
    l_rev, l_1s, l_5s = sums(late)

    e_1_rate = (e_1s / e_rev) if e_rev else None
    l_1_rate = (l_1s / l_rev) if l_rev else None

    # Recent negative keywords (last k months, ratings <= 2)
    last_months = [r["ym"] for r in late]
    neg_recent = (
        prod_df
        .withColumn("ym", F.date_trunc("month", F.to_timestamp("review_date")))
        .filter((F.col("ym").isin(last_months)) & (F.col("rating") <= 2))
        .select("review_text")
        .withColumn("text_lower", F.lower(F.coalesce(F.col("review_text"), F.lit(""))))
        .withColumn("word", F.explode(F.split(F.col("text_lower"), r"[^a-z]+")))
        .filter(F.length("word") >= 3)
    )

    stop = {"the","and","for","with","this","that","have","was","are","you","but","not","its","had","from",
            "they","has","one","two","get","got","use","used","very","after","still","when","why","how","who",
            "your","our","their","can","cannot","cant","did","didnt","does","doesnt","wasnt","werent","dont","wont",
            "much","more","less","than","then","into","out","over","under","been","being","make","made","could",
            "would","should","also","just","like","see","bad","good","okay","poor","well"}
    neg_recent = neg_recent.filter(~F.col("word").isin(list(stop)))

    top_terms_df = neg_recent.groupBy("word").count().orderBy(F.desc("count")).limit(5)
    top_terms = [r["word"] for r in top_terms_df.collect()] if top_terms_df.count() > 0 else []

    # Show monthly table (transparency)
    print("Monthly metrics:")
    monthly.show(n, truncate=False)

    # --- Format values safely BEFORE f-string ---
    def fmt_num(x, digits=2):
        return f"{x:.{digits}f}" if x is not None else "n/a"

    def fmt_pct(x):
        return f"{x*100:.1f}%" if x is not None else "n/a"

    first_m = first_month.strftime("%Y-%m") if first_month else "n/a"
    last_m  = last_month.strftime("%Y-%m") if last_month else "n/a"
    first_avg_txt = fmt_num(first_avg, 2)
    last_avg_txt  = fmt_num(last_avg, 2)
    drop_txt      = fmt_num(rating_drop_val, 2)
    e_rate_txt    = fmt_pct(e_1_rate)
    l_rate_txt    = fmt_pct(l_1_rate)
    kw            = ", ".join(top_terms[:3]) if top_terms else "no dominant terms"

    analysis = (
        f"{product_name}: The average rating fell from {first_avg_txt} in {first_m} to {last_avg_txt} in {last_m} "
        f"(drop {drop_txt}). Review volume shifted from {e_rev} (early window) to {l_rev} (late window). "
        f"The share of 1★ reviews increased from {e_rate_txt} to {l_rate_txt}. "
        f"Recent negative comments commonly mention {kw}. "
        f"Recommendation: address the top issues via QA checks, clearer setup guidance, and proactive support; "
        f"track monthly ratings until recovery is sustained."
    )

    print("\n--- Suggested ≤150‑word analysis ---\n")
    print(analysis)

Monthly metrics:
+-------------------+-------+----------+--------+---------+
|ym                 |reviews|avg_rating|one_star|five_star|
+-------------------+-------+----------+--------+---------+
|2016-04-01 00:00:00|1      |5.0       |0       |1        |
|2016-05-01 00:00:00|2      |5.0       |0       |2        |
|2016-06-01 00:00:00|13     |4.846     |0       |11       |
|2016-07-01 00:00:00|19     |4.684     |0       |13       |
|2016-08-01 00:00:00|27     |4.148     |1       |12       |
|2016-09-01 00:00:00|63     |4.571     |0       |43       |
|2016-10-01 00:00:00|35     |4.457     |1       |20       |
|2016-11-01 00:00:00|30     |4.6       |0       |19       |
|2016-12-01 00:00:00|46     |4.478     |1       |30       |
|2017-01-01 00:00:00|92     |4.511     |1       |62       |
|2017-02-01 00:00:00|37     |4.486     |0       |22       |
|2017-03-01 00:00:00|41     |4.61      |0       |28       |
|2017-04-01 00:00:00|32     |4.594     |0       |21       |
|2017-05-01 00:00:00|25

## Step 12 Performance Botteleneck and Optimization
After implementing all the above analytical queries, identify a query
where there is a performance bottleneck in your Spark job and propose
and implement optimizations. Compare the execution times before and
after optimization, and justify your approach.

In [ ]:
# Performance bottleneck & optimization
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

# Rebuild monthly averages
monthly_avg = (
    df_q
    .filter(F.col("review_date").isNotNull() & F.col("name").isNotNull())
    .withColumn("year_month", F.date_trunc("month", F.to_timestamp("review_date")))
    .groupBy("name", "year_month")
    .agg(F.round(F.avg("rating"), 3).alias("avg_rating"))
)

# ------------------------
# Baseline Method
# ------------------------
spark.conf.set("spark.sql.shuffle.partitions", "200")  # typical default
w_full = Window.partitionBy("name").orderBy("year_month") \
               .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

t0 = time.perf_counter()
baseline = (
    monthly_avg
    .withColumn("first_month", F.first("year_month").over(w_full))
    .withColumn("last_month",  F.last("year_month").over(w_full))
    .withColumn("first_avg",   F.first("avg_rating", ignorenulls=True).over(w_full))
    .withColumn("last_avg",    F.last("avg_rating",  ignorenulls=True).over(w_full))
    .select("name", "first_month", "last_month", "first_avg", "last_avg")
    .dropDuplicates(["name"])
    .withColumn("rating_drop", F.round(F.col("first_avg") - F.col("last_avg"), 3))
    .filter(F.col("rating_drop") > 0)
    .orderBy(F.desc("rating_drop"), F.asc("name"))
    .limit(10)
)
baseline_result = baseline.collect()  # action to materialize
t1 = time.perf_counter()
baseline_time = t1 - t0

print(f"Baseline completed in {baseline_time:.3f} sec with {len(baseline_result)} rows.")

# ---------------------------------------------
# Optimized: fewer shuffle partitions + caching
#            + repartition by key + array method
# ---------------------------------------------
spark.conf.set("spark.sql.shuffle.partitions", "64")

monthly_avg_cached = monthly_avg.repartition("name").persist(StorageLevel.MEMORY_AND_DISK)

# Warm-up touch (optional): count once so cache is populated
_ = monthly_avg_cached.count()

t2 = time.perf_counter()
opt = (
    monthly_avg_cached
    .groupBy("name")
    .agg(
        F.min("year_month").alias("first_month"),
        F.max("year_month").alias("last_month"),
        # Sort (month, rating) so we can take first/last without window
        F.array_sort(F.collect_list(F.struct("year_month", "avg_rating"))).alias("arr")
    )
    .withColumn("first_avg", F.element_at(F.col("arr"), 1).getItem("avg_rating"))
    .withColumn("last_avg",  F.element_at(F.col("arr"), -1).getItem("avg_rating"))
    .drop("arr")
    .withColumn("rating_drop", F.round(F.col("first_avg") - F.col("last_avg"), 3))
    .filter(F.col("rating_drop") > 0)
    .orderBy(F.desc("rating_drop"), F.asc("name"))
    .limit(10)
)
opt_result = opt.collect()  # action
t3 = time.perf_counter()
optimized_time = t3 - t2

print(f"Optimized completed in {optimized_time:.3f} sec with {len(opt_result)} rows.")

# Show both results (optional)
print("\nTop 10 (optimized):")
for r in opt_result:
    print(r)

Baseline completed in 1.204 sec with 10 rows.
Optimized completed in 0.768 sec with 10 rows.

Top 10 (optimized):
Row(name='Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders', first_month=datetime.datetime(2017, 3, 1, 0, 0), last_month=datetime.datetime(2017, 7, 1, 0, 0), first_avg=4.75, last_avg=1.0, rating_drop=3.75)
Row(name='Kindle Fire HDX 8.9"', first_month=datetime.datetime(2013, 11, 1, 0, 0), last_month=datetime.datetime(2015, 3, 1, 0, 0), first_avg=4.0, last_avg=1.0, rating_drop=3.0)
Row(name='Alexa Voice Remote for Amazon Echo and Echo Dot', first_month=datetime.datetime(2016, 5, 1, 0, 0), last_month=datetime.datetime(2016, 12, 1, 0, 0), first_avg=3.0, last_avg=1.0, rating_drop=2.0)
Row(name='Kindle for Kids Bundle with the latest Kindle E-reader', first_month=datetime.datetime(2016, 10, 1, 0, 0), last_month=datetime.datetime(2017, 4, 1, 0, 0), first_avg=5.0, last_avg=3.0, rating_drop=2.0)
Row(name='Replacement Remote for Amazon Fire TV